# MVA Hackathon 2026 - Track 2: drug-repurposing hypotheses

Runs the Track 2 analysis (`protocol/addendum_four_lines.md` and `protocol/addendum_review2.md`) on Google Colab, on a CPU runtime. Hypotheses only: not medical advice, no dosing, FDA-approved drugs only.

Run the cells in order. Downloads happen only in section 3 and only for the sources named there. Nothing here asks for passwords or keys; Drive access, if you use it, is the standard Colab prompt.

**Before you start.** Zip the repository folder `HSelith_MVA-Track2` and keep inside it `data/universe.json`, `data/dti_e1_all.json`, `data/sources/record.json` and `out/` (the rest of `data/` is downloaded or not needed here; leave out `data/cache`, `data/private`, `.git`, `__pycache__`).
Either put the zip at `MyDrive/mva_track2/HSelith_MVA-Track2.zip` (`USE_DRIVE = True`) or upload it to `/content/HSelith_MVA-Track2.zip` with the file panel (`USE_DRIVE = False`).

**What this notebook checks.** (1) The 169 tests on Linux. (2) That the source files download with the SHA-256 recorded in `data/sources/record.json`. (3) That two results of the final analysis are regenerated byte for byte (compared with `out/checksums.sha256`). The full analysis (stages 17 to 21 of `run_all`, several hours) is described in section 5 and is not run here.

## 1. Environment and repository

In [ ]:
import os, platform, shutil, subprocess, sys, zipfile


def sh(cmd, check=True):
    """Run a shell command, stream its output and stop on failure."""
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise SystemExit(f"command failed ({p.returncode}): {cmd}")
    return p.stdout


print(platform.platform()); print(sys.version)
sh("nproc; free -g; df -h /content", check=False)
sh("nvidia-smi -L || echo 'no GPU attached (fine: lines A to D are CPU)'", check=False)

In [ ]:
USE_DRIVE = True   # False: upload the zip to /content/HSelith_MVA-Track2.zip instead of using Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = "/content/drive/MyDrive/mva_track2/HSelith_MVA-Track2.zip"
    RESULTS = "/content/drive/MyDrive/mva_track2/results"   # results are copied here (Colab disks are erased)
else:
    ZIP = "/content/HSelith_MVA-Track2.zip"
    RESULTS = "/content/results"
assert os.path.exists(ZIP), "put the zip at " + ZIP
os.makedirs("/content/work", exist_ok=True)
zipfile.ZipFile(ZIP).extractall("/content/work")
REPO = "/content/work/HSelith_MVA-Track2"
assert os.path.isdir(REPO), "the zip must contain the folder HSelith_MVA-Track2"
os.makedirs(RESULTS, exist_ok=True)
os.chdir(REPO)
os.environ["PYTHONPATH"] = "src"
print("repository at", REPO)

In [ ]:
sh("pip install -q -r requirements-track2.lock")
sh("python -m pytest -q -p no:cacheprovider")

In [ ]:
sh("python -m track2.run_all --list")

## 2. Record the environment (for replication)

In [ ]:
sh("python --version > /content/environment.txt && pip freeze >> /content/environment.txt")
shutil.copy("/content/environment.txt", f"{RESULTS}/environment_colab.txt")

## 3. Download the sources (one source at a time)

`--list` shows what exists and its size without downloading. Each file is checked against the publisher's checksum when one is published and recorded with its SHA-256 in `data/sources/record.json`. The expected hashes shipped in the zip are compared with the ones just downloaded.

In [ ]:
sh("python -m track2.fetch_sources --list")
shutil.copy("data/sources/record.json", "/content/record_expected.json")   # hashes recorded when the analysis was run

In [ ]:
# Small sources (HPO, MONDO) and one large source (PrimeKG, about 395 MB). Add depmap_crispr and depmap_covariates for the full analysis.
SOURCES = ["hpo_annotations", "hpo_ontology", "mondo", "primekg"]
sh("python -m track2.fetch_sources --only " + " ".join(SOURCES))

In [ ]:
import json
exp = json.load(open("/content/record_expected.json")); got = json.load(open("data/sources/record.json"))
bad = [k for k in got if k in exp and exp[k]["sha256"] != got[k]["sha256"]]
new = [k for k in got if k not in exp]
print("files compared:", len([k for k in got if k in exp]), "| different SHA-256:", bad, "| not in the expected record:", new)
assert not bad, "a source changed since the analysis was run (MONDO 'latest' can do this): the results may differ"

In [ ]:
os.makedirs(f"{RESULTS}/sources", exist_ok=True)
sh(f"cp -r data/sources/* {RESULTS}/sources/ ; du -sh {RESULTS}/sources")

## 4. Regenerate results and compare them with the recorded checksums

Two results of the final analysis are regenerated into a fresh folder and compared with `out/checksums.sha256`: the phenotype module (E5) and the patient query of line A without the parental term (E5, needs PrimeKG from section 3).

In [ ]:
import hashlib
os.makedirs("/content/rep", exist_ok=True)
sh("python -m track2.phenotype_module --out /content/rep/phenotype_module_e5.json")
sh("python -m track2.kg_query --out /content/rep --e5")
record = dict(line.split("  ", 1)[::-1] for line in open("out/checksums.sha256").read().splitlines() if line and not line.startswith("#"))
for fresh, recorded in (("/content/rep/phenotype_module_e5.json", "out/phenotype_module_e5.json"), ("/content/rep/kg_query_e5.json", "out/kg_query_e5.json")):
    digest = hashlib.sha256(open(fresh, "rb").read()).hexdigest()
    print(recorded, "-> identical to the recorded hash:", digest == record[recorded])
    assert digest == record[recorded], "not reproduced byte for byte"

In [ ]:
sh("python -m track2.run_all --only 8")      # consensus stage (fast)
sh("python -m track2.run_all --only 22")     # verify every recorded checksum

## 5. The full analysis (not run in this notebook)

Stages 17 to 21 of `python -m track2.run_all` reproduce the final analysis: sources (17), line A (18), line B (19), lines C and D (20), combination, class analysis, caution list, age tables, report, methods, form and PDF (21). They need the earlier stages' inputs (`data/universe.json`, the drug-target links and the STRING and ChEMBL caches, produced by stages 1 to 16) and take several hours on CPU; line C alone uses four worker processes. Use `--dry-run` to see the commands and `--from-stage N` to resume.

The Drive mount, if you use it, must be authorised interactively in a browser; it cannot be tested from a script.

## 6. Save the results

In [ ]:
sh(f"mkdir -p {RESULTS}/rep && cp -r /content/rep/* {RESULTS}/rep/ && cp -r out {RESULTS}/ && ls {RESULTS}")